# Remaining useful life prediction with C-MAPSS

RUL is a time-to-event profile: the event is end of useful life and the target
is remaining operating cycles. `pdmdata.tasks.rul.prepare_cmapss` builds a
Polars-tabular experiment bundle from the source-faithful C-MAPSS loader.

## Learning goals

- prepare train / validation / test with unit-level hold-out;
- fit a row-level regressor without leaking across engines;
- score the official last-observation test protocol (MAE, RMSE, NASA score).

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata
from sklearn.ensemble import HistGradientBoostingRegressor
from pdmdata.tasks.rul import evaluate_test_predictions, prepare_cmapss

In [ ]:
pdmdata.download("cmapss")
bundle = prepare_cmapss(
    "FD001",
    rul_cap=125,
    validation_fraction=0.2,
    random_state=0,
    drop_constant_features=True,
)
bundle.summary()

In [ ]:
bundle.train.frame.select(
    "unit_number", "cycle", "RUL", "time_to_event", "event_observed"
).head()

In [ ]:
model = HistGradientBoostingRegressor(
    loss="absolute_error", random_state=0
)
X_train, y_train, _ = bundle.train.to_numpy()
X_val, y_val, _ = bundle.validation.to_numpy()
model.fit(X_train, y_train)
validation_prediction = np.maximum(model.predict(X_val), 0)
from pdmdata.tasks.rul import regression_metrics
print(regression_metrics(y_val, validation_prediction))

In [ ]:
X_all, y_all, _ = prepare_cmapss(
    "FD001",
    rul_cap=125,
    validation_fraction=0.0,
    drop_constant_features=True,
).train.to_numpy()
model.fit(X_all, y_all)
eval_split = bundle.test_eval_split()
prediction = np.maximum(model.predict(eval_split.X().to_numpy()), 0)
print(evaluate_test_predictions(bundle, prediction))

## Interpretation and limitations

`prepare_cmapss` keeps loaders source-faithful and applies RUL construction,
optional piecewise capping, constant-feature dropping, and unit hold-out in the
task layer. Uncapped RUL counts cycles to failure; it does not mark fault onset.
Report `rul_cap` and the NASA score because both change the task materially.
Test targets stay out of `bundle.test`; use `test_eval_split()` for the official
one-row-per-engine protocol.